In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
import json
import os
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import random
from PIL import Image
from torch.utils.data import Dataset
from typing import Tuple, Dict, List
import torch
from torchvision import transforms
import torchvision.models as models
from timeit import default_timer as timer
from sklearn.model_selection import KFold
from tqdm.auto import tqdm
from transformers import BertModel, BertTokenizer, ViTModel

In [3]:
def read_jsonl(file_path):
    data = []
    with open(file_path, 'r') as file:
        for line in file:
            data.append(json.loads(line))
    return data

train_data = read_jsonl('/content/drive/MyDrive/hateful_memes/train.jsonl')

In [4]:
def create_dataframe(data):
    df = pd.DataFrame(data)
    return df

df = create_dataframe(train_data)

In [5]:
df

,id,img,label,text
0,42953,img/42953.png,0,its their character not their color that matters
1,23058,img/23058.png,0,don't be afraid to love again everyone is not ...
2,13894,img/13894.png,0,putting bows on your pet
3,37408,img/37408.png,0,i love everything and everybody! except for sq...
4,82403,img/82403.png,0,"everybody loves chocolate chip cookies, even h..."
...,...,...,...,...
8495,10423,img/10423.png,1,nobody wants to hang auschwitz me
8496,98203,img/98203.png,1,when god grants you a child after 20 years of ...
8497,36947,img/36947.png,1,gays on social media: equality! body positivit...
8498,16492,img/16492.png,1,having a bad day? you could be a siamese twin ...


In [9]:
input_dir = '/content/drive/MyDrive/hateful_memes/'
output_dir = '/content/drive/MyDrive/hateful_memes/'

In [7]:
import albumentations as A
from PIL import Image
import numpy as np
import os
import pandas as pd

# Define the augmentation pipeline
augmentations = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=30, p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussianBlur(p=0.3),
])

/usr/local/lib/python3.10/dist-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 1.4.18 (you have 1.4.15). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [8]:
df['label'].value_counts()

,count
label,
0,5481
1,3019


In [10]:
augmented_data = []
# Set the desired number of augmented images
desired_augmented_count = 2500
augmented_count = 0  # Counter for augmented images

# Augment each image in class 1
for idx, row in df.iterrows():
    # Check if the label is 1
    if row['label'] == 1:
        img_path = os.path.join(input_dir, row['img'])
        img = np.array(Image.open(img_path))

        # Perform augmentation
        augmented = augmentations(image=img)['image']
        augmented_image = Image.fromarray(augmented)

        # Create new image ID and name
        new_image_id = f"aug_{row['id']}_{augmented_count}"
        new_image_name = f"augmented_img/aug_{row['id']}_{augmented_count}.png"

        # Save the augmented image
        augmented_image.save(os.path.join(output_dir, new_image_name))

        # Add new entry to the augmented data (new id and name, but same label and text)
        augmented_data.append({
            'id': new_image_id,
            'img': new_image_name,
            'label': row['label'],  # Same label
            'text': row['text']     # Same text
        })

        augmented_count += 1  # Increment the counter

        # Break the loop if the desired count is reached
        if augmented_count >= desired_augmented_count:
            break

In [11]:
# Convert augmented data to DataFrame
augmented_df = pd.DataFrame(augmented_data)

In [12]:
# Concatenate original dataframe and augmented dataframe
final_df = pd.concat([df, augmented_df], ignore_index=True)

In [13]:
final_df

,id,img,label,text
0,42953,img/42953.png,0,its their character not their color that matters
1,23058,img/23058.png,0,don't be afraid to love again everyone is not ...
2,13894,img/13894.png,0,putting bows on your pet
3,37408,img/37408.png,0,i love everything and everybody! except for sq...
4,82403,img/82403.png,0,"everybody loves chocolate chip cookies, even h..."
...,...,...,...,...
10995,aug_58743_2495,augmented_img/aug_58743_2495.png,1,"i'm a parasite, also known as a zionist we zio..."
10996,aug_02719_2496,augmented_img/aug_02719_2496.png,1,your black neighbor after you called the cops
10997,aug_47369_2497,augmented_img/aug_47369_2497.png,1,my favorite animal is carrot
10998,aug_68517_2498,augmented_img/aug_68517_2498.png,1,excuse me goy do you have a minute to talk abo...


In [14]:
# Shuffle the DataFrame
shuffled_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [15]:
shuffled_df

,id,img,label,text
0,93072,img/93072.png,1,"black people black people ""why would white ame..."
1,90638,img/90638.png,0,i think it's great when women succeed at work ...
2,20957,img/20957.png,0,jew haters demonrat sewage an evil cancer that...
3,31278,img/31278.png,1,"it's not a hijab it's a diaper, for peoples wi..."
4,79368,img/79368.png,1,how do you start a rave in ethiopia tape a pei...
...,...,...,...,...
10995,75023,img/75023.png,1,"shoot boy, i ain't racist i've got four black ..."
10996,38509,img/38509.png,0,1947 something went... 2017... terribly right
10997,28096,img/28096.png,0,this is for all you nosey fuckers that only fo...
10998,81720,img/81720.png,0,let's go! staring contest!


In [16]:
# Save the updated dataframe
final_df.to_csv('/content/drive/MyDrive/hateful_memes/updated_training_data.csv', index=False)
# Save the shuffled DataFrame (optional)
shuffled_df.to_csv('/content/drive/MyDrive/hateful_memes/shuffled_training_data.csv', index=False)

In [17]:
shuffled_df['label'].value_counts()

,count
label,
1,5519
0,5481
